# 中文 OCR 识别

使用 EasyOCR 识别 `photos_in/` 目录中的中文图像，支持长截图自动分段处理。

In [1]:
# 安装依赖
# 第1步：装 CUDA 版 PyTorch（5090 需要 cu130，必须先装这个）
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
# 第2步：装 easyocr
%pip install easyocr Pillow

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import easyocr
import os
from pathlib import Path
from PIL import Image
import numpy as np

# 初始化 EasyOCR reader（首次运行会下载模型）
# gpu=True 需要 CUDA 支持，没有显卡设为 False
reader = easyocr.Reader(['ch_sim', 'en'], gpu=True)

INPUT_DIR = Path('photos_in')
OUTPUT_DIR = Path('ocr_output')
OUTPUT_DIR.mkdir(exist_ok=True)

SUPPORTED_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tiff', '.tif'}

# 长截图分段阈值：高度超过此值（像素）时自动分段
LONG_IMAGE_THRESHOLD = 2000
# 每段高度
SEGMENT_HEIGHT = 1500
# 段间重叠像素，避免文字被截断
OVERLAP = 150

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [3]:
def split_long_image(img: Image.Image) -> list[np.ndarray]:
    """将长截图按高度分段，段间有重叠以避免截断文字。"""
    w, h = img.size
    if h <= LONG_IMAGE_THRESHOLD:
        return [np.array(img)]

    segments = []
    y = 0
    while y < h:
        y_end = min(y + SEGMENT_HEIGHT, h)
        crop = img.crop((0, y, w, y_end))
        segments.append(np.array(crop))
        y += SEGMENT_HEIGHT - OVERLAP
    
    print(f'  长截图 ({w}x{h})，分为 {len(segments)} 段处理')
    return segments


def ocr_image(image_path: Path) -> str:
    """对单张图片做 OCR，长截图自动分段后合并结果。"""
    img = Image.open(image_path).convert('RGB')
    segments = split_long_image(img)

    all_lines = []
    for i, seg in enumerate(segments):
        results = reader.readtext(seg, detail=0, paragraph=True)
        all_lines.extend(results)

    # 去除因重叠区域产生的重复行
    deduped = []
    for line in all_lines:
        line = line.strip()
        if line and (not deduped or line != deduped[-1]):
            deduped.append(line)

    return '\n'.join(deduped)

In [4]:
# 批量处理 photos_in 中所有图片
image_files = sorted(
    f for f in INPUT_DIR.iterdir()
    if f.suffix.lower() in SUPPORTED_EXT
)

print(f'共找到 {len(image_files)} 张图片\n')

results = {}
for img_path in image_files:
    print(f'处理: {img_path.name} ...')
    text = ocr_image(img_path)
    results[img_path.name] = text

    # 保存单张结果到 txt
    out_file = OUTPUT_DIR / f'{img_path.stem}.txt'
    out_file.write_text(text, encoding='utf-8')
    print(f'  -> 已保存: {out_file}')
    print(f'  识别到 {len(text)} 个字符\n')

print('全部完成！')

共找到 3 张图片

处理: 4d1760ed9243165f588559fe81c44e80.png ...
  长截图 (1945x29049)，分为 22 段处理


c:\Users\ovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
# 查看识别结果
for name, text in results.items():
    print(f'===== {name} =====')
    print(text[:2000])  # 只显示前2000字符
    if len(text) > 2000:
        print(f'\n... (共 {len(text)} 字符，完整内容见 ocr_output/{Path(name).stem}.txt)')
    print()